# Lab 3 - Hugging Face: Qwen + LoRA (train, evaluate, benchmark)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hyperscaleailabs/ml-platform-engineering/blob/main/notebooks/03_hf_qwen_lora.ipynb)

**Runtime:** ~20 minutes on a laptop CPU (measured: Apple M4, default settings); a Colab T4
GPU is several times faster. Downloads ~1 GB (model + dataset), cached after the first run.
**Recommended:** Colab -> Runtime -> Change runtime type -> T4 GPU.
**In a hurry?** `LAB_N_EVAL=64 LAB_MAX_STEPS=60` finishes in about 8 minutes and every
check still holds - the accuracy numbers are simply lower.

Lab 2 built a transformer block from scratch. This lab takes a *pretrained* one -
Qwen2.5-0.5B-Instruct, the same RMSNorm/RoPE/GQA/SwiGLU architecture you implemented - and adapts
it to a new task with **LoRA**, then evaluates and benchmarks it like a system you have to operate.

The framing matters. Fine-tuning is easy to make *look* like it worked. Most of this notebook is
about the parts that are hard to fake:

* an evaluation that cannot be gamed by output formatting,
* a baseline strong enough that beating it means something,
* benchmarks that report latency, throughput, memory and disk - not just accuracy.

## What you will do

1. Load Qwen2.5-0.5B-Instruct and map its modules onto the components from Lab 2.
2. Define an objective task: 6-way emotion classification, scored by constrained label likelihood.
3. Measure a **zero-shot and few-shot baseline**, including how often the model ignores the format.
4. Implement LoRA from scratch, verify it against PEFT numerically, then use PEFT.
5. Train with masked loss (completion-only) using the HuggingFace `Trainer`.
6. Re-evaluate: accuracy, per-class recall, confusion matrix.
7. Benchmark: trainable parameters, optimizer memory, adapter size on disk, inference latency
   merged vs unmerged, and throughput against batch size.

## 0. Setup

In [ ]:
import os
import sys
import json
import math
import time
import gc
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Colab ships torch; the rest is a ~30 s install.
    os.system(f"{sys.executable} -m pip install -q "
              f"'transformers>=4.44' 'peft>=0.12' 'datasets>=2.20' 'accelerate>=0.33'")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import transformers
import peft
import datasets as hfdatasets
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

# ---- Config (override via environment variables) -------------------------------------------------
MODEL_ID = os.environ.get("LAB_MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
SEED = int(os.environ.get("LAB_SEED", 0))
N_TRAIN = int(os.environ.get("LAB_N_TRAIN", 2000))     # training examples sampled from the dataset
N_EVAL = int(os.environ.get("LAB_N_EVAL", 300))        # held-out examples for every evaluation
MAX_STEPS = int(os.environ.get("LAB_MAX_STEPS", 200))
BATCH_SIZE = int(os.environ.get("LAB_BATCH_SIZE", 8))
GRAD_ACCUM = int(os.environ.get("LAB_GRAD_ACCUM", 2))
LORA_R = int(os.environ.get("LAB_LORA_R", 16))
LORA_ALPHA = int(os.environ.get("LAB_LORA_ALPHA", 32))
LEARNING_RATE = float(os.environ.get("LAB_LR", 2e-4))
EVAL_BATCH = int(os.environ.get("LAB_EVAL_BATCH", 8))
# --------------------------------------------------------------------------------------------------


def pick_device() -> torch.device:
    """CUDA if present, otherwise CPU. Override with LAB_DEVICE=cuda|mps|cpu.

    Note that Apple's MPS backend is NOT the default here, unlike in Labs 1 and 2. A 0.5B model in
    fp32 with long prompts reliably triggers `command buffer exited with error status / Internal
    Error` from the Metal driver on some macOS versions - a driver-level failure that PyTorch
    cannot catch or retry. Labs 1 and 2 work with far smaller tensors and are fine on MPS.
    If yours is healthy, opt back in with LAB_DEVICE=mps; it is several times faster than CPU.
    """
    requested = os.environ.get("LAB_DEVICE")
    if requested:
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = pick_device()

# Load the base weights in fp32 and let the Trainer handle mixed precision. Loading directly in
# fp16 and training on top of it is a well-known source of NaN losses: the master weights need
# more precision than the forward pass does. AMP keeps fp32 master weights and casts per-op.
USE_FP16 = DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported()
USE_BF16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()


# transformers 5 renamed `torch_dtype` to `dtype` and moved `rope_theta` under `rope_parameters`.
# Colab and local environments are frequently on different majors, so resolve both explicitly
# rather than discovering it as a TypeError halfway through the notebook.
TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"


def load_causal_lm(model_id, dtype=torch.float32, **kwargs):
    return AutoModelForCausalLM.from_pretrained(model_id, **{DTYPE_KW: dtype}, **kwargs)


def get_rope_theta(config) -> float:
    theta = getattr(config, "rope_theta", None)
    if theta is None:
        theta = (getattr(config, "rope_parameters", None) or {}).get("rope_theta", 10_000.0)
    return float(theta)


def set_seed(seed: int = SEED) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)


def check(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(f"CHECK FAILED: {message}")
    print(f"  ok - {message}")


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


set_seed()
print(f"transformers {transformers.__version__}  (major {TF_MAJOR}, dtype kwarg: {DTYPE_KW!r})")
print(f"peft         {peft.__version__}")
print(f"datasets     {hfdatasets.__version__}")
print(f"device       {DEVICE}   (bf16={USE_BF16}, fp16={USE_FP16})")
if DEVICE.type == "cpu":
    print("\nNOTE: running on CPU. Everything works, but expect roughly an hour. On Colab choose\n"
          "      Runtime -> Change runtime type -> T4 GPU. To go faster locally, lower\n"
          "      LAB_MAX_STEPS / LAB_N_EVAL, or on Apple silicon try LAB_DEVICE=mps (see above).")

## 1. The model

Qwen2.5-0.5B-Instruct is 494M parameters, Apache-2.0 licensed, and small enough to fine-tune on a
free Colab GPU. Print the config and compare it against the model you wrote in Lab 2.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = load_causal_lm(MODEL_ID).to(DEVICE)
base_model.eval()

cfg = base_model.config
print(f"{'hidden_size':<26} {cfg.hidden_size}")
print(f"{'num_hidden_layers':<26} {cfg.num_hidden_layers}")
print(f"{'num_attention_heads':<26} {cfg.num_attention_heads}")
print(f"{'num_key_value_heads':<26} {cfg.num_key_value_heads}   <- GQA: {cfg.num_attention_heads // cfg.num_key_value_heads} query heads share each KV head")
print(f"{'intermediate_size':<26} {cfg.intermediate_size}   <- SwiGLU hidden width")
print(f"{'vocab_size':<26} {cfg.vocab_size:,}")
print(f"{'rope_theta':<26} {get_rope_theta(cfg):,.0f}")
print(f"{'tie_word_embeddings':<26} {cfg.tie_word_embeddings}")
print(f"{'max_position_embeddings':<26} {cfg.max_position_embeddings:,}")

total_params = sum(p.numel() for p in base_model.parameters())
embed_params = base_model.get_input_embeddings().weight.numel()
print(f"\ntotal parameters      {total_params / 1e6:8.1f}M")
print(f"embedding table       {embed_params / 1e6:8.1f}M  ({embed_params / total_params:.1%} of the model)")
print(f"transformer body      {(total_params - embed_params) / 1e6:8.1f}M")
print(f"fp32 weights on disk  {total_params * 4 / 1e9:8.2f} GB")

check(cfg.num_attention_heads % cfg.num_key_value_heads == 0, "GQA head groups divide evenly")
check(cfg.tie_word_embeddings, "Qwen2.5-0.5B ties input and output embeddings, as in Lab 2")

In [ ]:
# The module names below are exactly the components from Lab 2. They are also the strings you will
# hand to LoRA as `target_modules`, so it is worth reading this list carefully.
print("One decoder layer:\n")
print(base_model.model.layers[0])

print("\nParameter shapes in layer 0:")
for name, param in base_model.model.layers[0].named_parameters():
    print(f"  {name:<28} {tuple(param.shape)}")

d_head = cfg.hidden_size // cfg.num_attention_heads
print(f"\nhead dim = {cfg.hidden_size} / {cfg.num_attention_heads} = {d_head}")
print(f"k_proj output = {cfg.num_key_value_heads} KV heads x {d_head} = {cfg.num_key_value_heads * d_head}")
check(base_model.model.layers[0].self_attn.k_proj.weight.shape[0] == cfg.num_key_value_heads * d_head,
      "the K projection is sized by KV heads, not query heads - that is GQA in the weights")

## 2. The task

**Emotion classification** on `dair-ai/emotion`: English tweets labelled with one of six emotions.
It is a good lab task for reasons that generalize to picking any fine-tuning benchmark:

* **Objective.** Accuracy against a fixed label set. No LLM judge, no rubric, no ambiguity.
* **Genuinely hard for a 0.5B model.** `joy` vs `love` and `anger` vs `sadness` need more than
  keyword matching, so the baseline is meaningfully below ceiling and there is room to improve.
* **Small.** 16k training rows, ~1 MB.

If the download fails (offline or firewalled), a small embedded sample keeps the notebook running
so you can still follow the mechanics.

In [ ]:
LABELS = ["sadness", "joy", "love", "anger", "fear", "surprise"]

FALLBACK_ROWS = [
    ("i didnt feel humiliated", 0), ("i feel so blessed to have such good friends", 1),
    ("i am feeling romantic and tender toward him", 2), ("i feel furious that they lied to me", 3),
    ("i feel terrified about the exam tomorrow", 4), ("i feel shocked that it happened so fast", 5),
    ("i just feel really hopeless and lost", 0), ("i feel great about how the day went", 1),
    ("i feel such deep affection for my sister", 2), ("i am feeling irritated and angry at work", 3),
    ("i feel scared walking home in the dark", 4), ("i feel amazed by the sudden news", 5),
] * 60


def load_emotion():
    try:
        ds = hfdatasets.load_dataset("dair-ai/emotion", "split")
        return ds["train"], ds["test"]
    except Exception as exc:
        print(f"dataset download failed ({exc}); using the embedded fallback sample")
        rows = {"text": [t for t, _ in FALLBACK_ROWS], "label": [l for _, l in FALLBACK_ROWS]}
        full = hfdatasets.Dataset.from_dict(rows).shuffle(seed=SEED)
        split = full.train_test_split(test_size=0.3, seed=SEED)
        return split["train"], split["test"]


train_raw, test_raw = load_emotion()
set_seed()
train_ds = train_raw.shuffle(seed=SEED).select(range(min(N_TRAIN, len(train_raw))))
eval_ds = test_raw.shuffle(seed=SEED).select(range(min(N_EVAL, len(test_raw))))

print(f"train pool {len(train_raw):,} -> using {len(train_ds):,}")
print(f"eval  pool {len(test_raw):,} -> using {len(eval_ds):,}\n")

eval_labels = np.array(eval_ds["label"])
counts = np.bincount(eval_labels, minlength=len(LABELS))
majority = counts.max() / counts.sum()
print(f"{'label':<10} {'eval count':>11} {'share':>8}")
for i, name in enumerate(LABELS):
    print(f"{name:<10} {counts[i]:>11} {counts[i] / counts.sum():>7.1%}")
print(f"\nmajority-class baseline: {majority:.1%}   random baseline: {1 / len(LABELS):.1%}")

for row in train_ds.select(range(3)):
    print(f"\n[{LABELS[row['label']]:<9}] {row['text']}")

check(len(eval_ds) > 0 and len(train_ds) > 0, "both splits are non-empty")

### The prompt

Instruction-tuned models expect their chat template. Skipping `apply_chat_template` and
concatenating raw strings is a real and costly mistake - the model was post-trained on those exact
special tokens (`<|im_start|>`, `<|im_end|>` for Qwen) and drifts badly without them. The template
ships in the tokenizer, so use it for both training and inference, and keep the two identical.

In [ ]:
SYSTEM_PROMPT = (
    "You are an emotion classifier. Classify the emotion expressed in the text as exactly one of: "
    + ", ".join(LABELS)
    + ". Reply with only that single word."
)


def build_prompt(text: str, few_shot: list[tuple[str, str]] | None = None) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for shot_text, shot_label in (few_shot or []):
        messages.append({"role": "user", "content": shot_text})
        messages.append({"role": "assistant", "content": shot_label})
    messages.append({"role": "user", "content": text})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


example_prompt = build_prompt("i feel like i am drowning in work")
print(example_prompt)
print(f"\nprompt length: {len(tokenizer(example_prompt)['input_ids'])} tokens")

# Label tokenization matters for the scoring evaluation below - print it explicitly.
print(f"\n{'label':<10} {'tokens':>7}  ids")
for name in LABELS:
    ids = tokenizer(name, add_special_tokens=False)["input_ids"]
    print(f"{name:<10} {len(ids):>7}  {ids}")

check(example_prompt.endswith("<|im_start|>assistant\n"),
      "the template ends with the assistant turn open, ready for the model to continue")

## 3. Evaluation, done two ways

This is the part people get wrong, so we do it twice and compare.

**A. Free generation + parse.** Ask the model to answer, generate tokens, parse the first word.
This is what production sees, and it conflates two very different failures: not knowing the answer,
and not following the output format.

**B. Constrained label scoring.** For each of the six labels, compute
$\log p(\text{label} \mid \text{prompt})$ and take the argmax. This measures *only* what the model
believes, always returns a valid label, and is the standard method behind harnesses like
`lm-evaluation-harness`. It is also 6 forward passes per example instead of one generation.

The gap between A and B is the "format tax" - and it is exactly what a small amount of fine-tuning
removes first.

In [ ]:
def body_and_head(model):
    """Split a causal LM into (transformer body, output projection).

    This lets us run the expensive part once and apply the 151,936-way projection only at the few
    positions we actually score. Handles both a plain transformers model and a PEFT-wrapped one;
    returns (None, None) for an unfamiliar layout so the caller can fall back to full logits.
    """
    inner = model.get_base_model() if hasattr(model, "get_base_model") else model
    body = getattr(inner, "model", None)
    head = inner.get_output_embeddings() if hasattr(inner, "get_output_embeddings") else None
    if body is None or head is None or not callable(body):
        return None, None
    return body, head


@torch.no_grad()
def score_labels(model, texts, few_shot=None, batch_size=EVAL_BATCH, length_normalize=False,
                 use_body=True):
    """Predict by comparing log p(label | prompt) across the fixed label set.

    Right padding is safe here: with causal attention, tokens after the label cannot influence the
    label's logits. (Generation is the opposite case - it needs LEFT padding, see below.)
    """
    model.eval()
    label_ids = [tokenizer(lbl, add_special_tokens=False)["input_ids"] for lbl in LABELS]

    sequences, spans, owners = [], [], []
    for i, text in enumerate(texts):
        prompt_ids = tokenizer(build_prompt(text, few_shot), add_special_tokens=False)["input_ids"]
        for j, lid in enumerate(label_ids):
            sequences.append(prompt_ids + lid)
            spans.append((len(prompt_ids), len(prompt_ids) + len(lid)))
            owners.append((i, j))

    scores = np.zeros((len(texts), len(LABELS)), dtype=np.float64)
    pad_id = tokenizer.pad_token_id
    body, head = body_and_head(model) if use_body else (None, None)

    for start in range(0, len(sequences), batch_size):
        chunk = sequences[start:start + batch_size]
        chunk_spans = spans[start:start + batch_size]
        chunk_owners = owners[start:start + batch_size]
        width = max(len(s) for s in chunk)

        input_ids = torch.full((len(chunk), width), pad_id, dtype=torch.long)
        attention = torch.zeros((len(chunk), width), dtype=torch.long)
        for r, seq in enumerate(chunk):
            input_ids[r, :len(seq)] = torch.tensor(seq)
            attention[r, :len(seq)] = 1

        ids_dev, att_dev = input_ids.to(DEVICE), attention.to(DEVICE)

        if body is not None:
            # Transformer body only: (batch, seq, 896) instead of (batch, seq, 151936).
            hidden = body(input_ids=ids_dev, attention_mask=att_dev).last_hidden_state
            slice_logits = lambda r, s, e: head(hidden[r, s - 1:e - 1, :]).float()
        else:
            logits = model(input_ids=ids_dev, attention_mask=att_dev).logits
            slice_logits = lambda r, s, e: logits[r, s - 1:e - 1, :].float()

        for r, ((s, e), (i, j)) in enumerate(zip(chunk_spans, chunk_owners)):
            # logits at position t predict token t+1, so the token at index t is scored by
            # the projection of hidden state t-1.
            span = slice_logits(r, s, e)                           # (n_label_tokens, vocab)
            tok_lp = torch.log_softmax(span, dim=-1).gather(
                -1, ids_dev[r, s:e].unsqueeze(-1)).squeeze(-1)
            total = tok_lp.sum().item()
            scores[i, j] = total / (e - s) if length_normalize else total

    return scores.argmax(axis=1), scores


@torch.no_grad()
def generate_labels(model, texts, few_shot=None, batch_size=EVAL_BATCH, max_new_tokens=6):
    """Predict by generating and parsing. Returns (predictions, raw_outputs).

    Prediction is -1 when the output does not contain a valid label - we count those separately
    rather than hiding them, because 'unparseable' is a distinct and actionable failure.
    """
    model.eval()
    original_side = tokenizer.padding_side
    tokenizer.padding_side = "left"   # generation must pad on the LEFT, or the model continues pads
    preds, raws = [], []
    try:
        for start in range(0, len(texts), batch_size):
            chunk = texts[start:start + batch_size]
            prompts = [build_prompt(t, few_shot) for t in chunk]
            enc = tokenizer(prompts, return_tensors="pt", padding=True,
                            add_special_tokens=False).to(DEVICE)
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
            for row, seq in enumerate(out):
                text_out = tokenizer.decode(seq[enc["input_ids"].shape[1]:],
                                            skip_special_tokens=True).strip()
                raws.append(text_out)
                lowered = text_out.lower()
                match = next((k for k, lbl in enumerate(LABELS) if lowered.startswith(lbl)), None)
                if match is None:
                    match = next((k for k, lbl in enumerate(LABELS) if lbl in lowered), -1)
                preds.append(match)
    finally:
        tokenizer.padding_side = original_side
    return np.array(preds), raws

> **Why evaluation runs out of memory before training does.** A causal LM emits one logit per
> vocabulary entry per position. At batch 8, sequence 250 (the 6-shot prompt) and Qwen's
> 151,936-token vocabulary, a single fp32 logit tensor is
> `8 x 250 x 151936 x 4 bytes = 1.2 GB` - more than twice the size of the 494M-parameter model
> itself. Casting it or calling `log_softmax` on it allocates another copy each time.
>
> `score_labels` never builds that tensor. It runs the transformer **body** to get
> `(batch, seq, 896)` hidden states - about 7 MB - and applies the output projection only at the
> one or two positions per sequence that it actually scores. Same numbers, ~170x less memory, and
> substantially faster. Lab 4 takes the idea one step further and projects only the six
> vocabulary rows it needs to compare.

In [ ]:
def report(preds, gold, title):
    valid = preds >= 0
    acc = float((preds[valid] == gold[valid]).sum()) / len(gold)   # invalid counts as wrong
    print(f"{title:<34} accuracy {acc:6.2%}   unparseable {int((~valid).sum()):>3}/{len(gold)}")
    return acc

In [ ]:
# The body-only path is an optimization, so prove it changes nothing before relying on it for
# every number in this notebook.
_probe = list(eval_ds["text"])[:8]
_, scores_fast = score_labels(base_model, _probe, use_body=True)
_, scores_slow = score_labels(base_model, _probe, use_body=False)
print(f"max |body-only - full-logits| = {np.abs(scores_fast - scores_slow).max():.2e}")
check(np.allclose(scores_fast, scores_slow, atol=1e-3),
      "projecting only the scored positions gives the same log-probabilities as full logits")
check(np.array_equal(scores_fast.argmax(1), scores_slow.argmax(1)),
      "...and therefore the same predictions")

In [ ]:
eval_texts = list(eval_ds["text"])
t0 = time.time()
gen_preds_zs, gen_raw_zs = generate_labels(base_model, eval_texts)
acc_gen_zs = report(gen_preds_zs, eval_labels, "zero-shot, free generation")
print(f"({time.time() - t0:.1f}s)\n")

print("sample raw outputs - note how many are not a bare label:")
for text, raw in list(zip(eval_texts, gen_raw_zs))[:8]:
    print(f"  {raw!r:<30} <- {text[:52]}")

In [ ]:
t0 = time.time()
score_preds_zs, score_matrix_zs = score_labels(base_model, eval_texts)
acc_score_zs = report(score_preds_zs, eval_labels, "zero-shot, constrained scoring")
print(f"({time.time() - t0:.1f}s)")

### An aside: summed log-probability is biased by label length

`sadness`, `fear` and `surprise` tokenize to two tokens; `joy`, `love` and `anger` to one. Summing
log-probabilities over a label therefore penalizes the longer ones - every extra token contributes
another negative number. Dividing by token count is the usual fix, and it is a heuristic rather
than a correction: with a fixed label set and a uniform prior, the *sum* is the mathematically
correct posterior. Which one scores better is an empirical question, so measure it rather than
picking one on principle.

In [ ]:
norm_preds_zs, _ = score_labels(base_model, eval_texts, length_normalize=True)
acc_norm_zs = report(norm_preds_zs, eval_labels, "zero-shot, length-normalized")
print(f"{'zero-shot, summed (from above)':<34} accuracy {acc_score_zs:6.2%}")

one_token = {i for i, l in enumerate(LABELS)
             if len(tokenizer(l, add_special_tokens=False)["input_ids"]) == 1}
print(f"\nshare of predictions that are single-token labels (joy/love/anger):")
print(f"  summed          {np.mean([p in one_token for p in score_preds_zs]):.1%}")
print(f"  length-normed   {np.mean([p in one_token for p in norm_preds_zs]):.1%}")
print(f"  true share      {np.mean([g in one_token for g in eval_labels]):.1%}")

check(not np.array_equal(score_preds_zs, norm_preds_zs),
      "the two scoring rules genuinely disagree on some examples - this is a real choice, "
      "not a formality")

In [ ]:
FEW_SHOT = [("i didnt feel humiliated", "sadness"),
            ("i feel so blessed to have such good friends", "joy"),
            ("i am feeling romantic and tender toward him", "love"),
            ("i feel furious that they lied to me", "anger"),
            ("i feel terrified about the exam tomorrow", "fear"),
            ("i feel shocked that it happened so fast", "surprise")]

score_preds_fs, _ = score_labels(base_model, eval_texts, few_shot=FEW_SHOT)
acc_score_fs = report(score_preds_fs, eval_labels, "6-shot, constrained scoring")

gen_preds_fs, _ = generate_labels(base_model, eval_texts, few_shot=FEW_SHOT)
acc_gen_fs = report(gen_preds_fs, eval_labels, "6-shot, free generation")

print(f"\nmajority-class baseline            {majority:6.2%}")
check(acc_score_zs > 1 / len(LABELS), "constrained scoring beats random - the model knows something")
check(acc_score_zs >= acc_gen_zs,
      "constrained scoring is at least as good as parsing free generation "
      "(it cannot lose points to formatting)")

**Read the baseline table before continuing.** Few-shot prompting is a real baseline, and it costs
nothing but context. If LoRA does not clearly beat 6-shot prompting, the fine-tune was not worth
the training run, the extra artifact, and the extra thing to version and serve. Reporting a
fine-tune against a *zero-shot* baseline only is one of the most common ways published gains
evaporate in production.

## 4. LoRA

Full fine-tuning updates every weight $W \in \mathbb{R}^{d \times k}$. LoRA freezes $W$ and learns
a low-rank update:

$$W' = W + \Delta W = W + \frac{\alpha}{r} B A, \qquad B \in \mathbb{R}^{d\times r},\; A \in \mathbb{R}^{r \times k},\; r \ll \min(d,k)$$

$A$ is initialized randomly and $B$ to **zeros**, so $\Delta W = 0$ at step 0 - training starts
from exactly the pretrained model, no warm-up damage. The $\alpha/r$ scaling makes the effective
update magnitude roughly independent of $r$, so you can change rank without re-tuning the
learning rate. (This is why people quote $\alpha = 2r$ as a default.)

The premise is that the *update* needed to adapt a pretrained model to a task is intrinsically
low-rank, even though $W$ itself is not. Empirically this holds remarkably well.

### Why it matters for a platform, not just a GPU budget

| | Full fine-tune | LoRA (r=16) |
|---|---|---|
| Trainable parameters | 494M | ~1M |
| Adam optimizer state | 2 fp32 moments per parameter | per *adapter* parameter |
| Artifact per task | 1 GB+ | a few MB |
| Serving N tasks | N model copies | 1 base + N adapters, swappable per request |

The last row is the one that changes system design. Compute the memory arithmetic:

In [ ]:
def training_memory_gb(n_trainable, n_total, bytes_weight=4):
    """Weights + gradients + Adam's two moments. Activations are extra and batch-dependent."""
    weights = n_total * bytes_weight
    grads = n_trainable * bytes_weight
    adam = n_trainable * 2 * 4          # exp_avg and exp_avg_sq, always fp32
    return (weights + grads + adam) / 1e9


lora_trainable_est = 0
for layer in base_model.model.layers:
    for mod_name in ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]:
        mod = layer.self_attn if "proj" in mod_name and mod_name.startswith(("q", "k", "v", "o")) \
            else layer.mlp
        w = getattr(mod, mod_name).weight
        lora_trainable_est += LORA_R * (w.shape[0] + w.shape[1])

print(f"{'strategy':<22} {'trainable':>12} {'train memory':>14}")
print(f"{'full fine-tune':<22} {total_params:>12,} {training_memory_gb(total_params, total_params):>13.2f} GB")
print(f"{'LoRA r=' + str(LORA_R):<22} {lora_trainable_est:>12,} "
      f"{training_memory_gb(lora_trainable_est, total_params):>13.2f} GB")
print(f"\nLoRA trains {lora_trainable_est / total_params:.2%} of the parameters and needs "
      f"{training_memory_gb(total_params, total_params) / training_memory_gb(lora_trainable_est, total_params):.1f}x less memory.")

### Implement it, then check it against PEFT

Same pattern as Labs 1 and 2: write the thing, then prove your version matches the library's.

In [ ]:
class MyLoRALinear(nn.Module):
    """Minimal LoRA wrapper around a frozen nn.Linear."""

    def __init__(self, base: nn.Linear, r: int, alpha: int, dropout: float = 0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False           # the whole point: the base never moves

        self.r, self.scaling = r, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        self.dropout = nn.Dropout(dropout)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))   # matches PEFT's default init

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B) * self.scaling

    def merged_weight(self):
        return self.base.weight + (self.lora_B @ self.lora_A) * self.scaling


set_seed()
demo_base = nn.Linear(64, 128, bias=False)
mine = MyLoRALinear(demo_base, r=8, alpha=16)
x_demo = torch.randn(4, 64)

check(torch.allclose(mine(x_demo), demo_base(x_demo), atol=1e-6),
      "B initialized to zero means LoRA is an exact no-op at step 0")

# Now give B a non-zero value and confirm the merged weight is equivalent to the two-matmul form.
with torch.no_grad():
    mine.lora_B.normal_(0, 0.02)
check(torch.allclose(mine(x_demo), F.linear(x_demo, mine.merged_weight()), atol=1e-5),
      "the merged weight reproduces the adapter output exactly - merging is free at inference")

# Cross-check against PEFT by copying my A/B into a PEFT-wrapped copy of the same layer.
peft_layer = get_peft_model(
    nn.Sequential(nn.Linear(64, 128, bias=False)),
    LoraConfig(r=8, lora_alpha=16, target_modules=["0"], lora_dropout=0.0, bias="none"),
)
with torch.no_grad():
    tgt = peft_layer.base_model.model[0]
    tgt.base_layer.weight.copy_(demo_base.weight)
    tgt.lora_A["default"].weight.copy_(mine.lora_A)
    tgt.lora_B["default"].weight.copy_(mine.lora_B)

peft_layer.eval()
mine.eval()
with torch.no_grad():
    delta = (peft_layer(x_demo) - mine(x_demo)).abs().max().item()
print(f"max |PEFT - my implementation| = {delta:.2e}")
check(delta < 1e-5, "my LoRA implementation is numerically identical to PEFT's")

del peft_layer, mine, demo_base
free_memory()

### Attaching adapters to Qwen

`target_modules` decides where adapters go. Attention-only (`q_proj`, `v_proj`) is the original
paper's setting and the cheapest; including the MLP projections (`gate_proj`, `up_proj`,
`down_proj`) generally works better because the MLP holds most of the parameters. We take all
seven, which is the current common default.

In [ ]:
free_memory()
set_seed()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"\ntrainable {trainable:,}   frozen {frozen:,}   ratio {trainable / (trainable + frozen):.3%}")

check(trainable > 0 and trainable / (trainable + frozen) < 0.05,
      "under 5% of parameters are trainable")
check(all(not p.requires_grad for n, p in model.named_parameters() if "lora_" not in n),
      "every non-LoRA parameter is frozen")

# Adapters are a no-op before training, so accuracy must be bit-identical to the baseline.
preds_untrained, _ = score_labels(model, eval_texts[:32])
check(np.array_equal(preds_untrained, score_preds_zs[:32]),
      "the freshly-adapted model predicts exactly what the base model did (B=0)")

## 5. Training

### Loss masking is the part that matters

The training example is `prompt + label`. If you compute loss over the whole sequence, the model
spends most of its gradient learning to reproduce the system prompt and the tweet - text it will
always be *given*, never asked to produce. Set those positions to `-100` (PyTorch's
`ignore_index`) so the loss is computed on the completion only.

With a one-word completion and a ~90-token prompt, this is not a subtle effect: it is the
difference between ~1% of tokens carrying signal and ~99% of them being noise.

In [ ]:
def build_training_example(text: str, label_idx: int):
    prompt_ids = tokenizer(build_prompt(text), add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(LABELS[label_idx], add_special_tokens=False)["input_ids"]
    completion_ids = completion_ids + [tokenizer.eos_token_id]   # teach it to stop

    input_ids = prompt_ids + completion_ids
    labels = [-100] * len(prompt_ids) + completion_ids           # <- the mask
    return {"input_ids": input_ids, "labels": labels}


tokenized_train = [build_training_example(t, l)
                   for t, l in zip(train_ds["text"], train_ds["label"])]

sample = tokenized_train[0]
n_supervised = sum(1 for x in sample["labels"] if x != -100)
print(f"example length {len(sample['input_ids'])} tokens, supervised on {n_supervised}")
print(f"-> {n_supervised / len(sample['input_ids']):.1%} of positions carry loss")
print(f"\nsupervised tokens: {tokenizer.decode([x for x in sample['labels'] if x != -100])!r}")

check(len(sample["input_ids"]) == len(sample["labels"]), "inputs and labels are aligned 1:1")
check(sample["labels"][:len(sample["labels"]) - n_supervised] == [-100] * (len(sample["labels"]) - n_supervised),
      "every prompt position is masked out of the loss")


class CausalCollator:
    """Pad a batch to its longest member. Inputs pad with pad_token_id, labels pad with -100."""

    def __init__(self, pad_id: int):
        self.pad_id = pad_id

    def __call__(self, features):
        width = max(len(f["input_ids"]) for f in features)
        input_ids, labels, attention = [], [], []
        for f in features:
            gap = width - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_id] * gap)
            labels.append(f["labels"] + [-100] * gap)
            attention.append([1] * len(f["input_ids"]) + [0] * gap)
        return {"input_ids": torch.tensor(input_ids),
                "labels": torch.tensor(labels),
                "attention_mask": torch.tensor(attention)}


collator = CausalCollator(tokenizer.pad_token_id)
batch = collator(tokenized_train[:4])
print(f"\nbatch input_ids {tuple(batch['input_ids'].shape)}  "
      f"supervised positions {(batch['labels'] != -100).sum().item()} / {batch['labels'].numel()}")
check((batch["labels"][batch["attention_mask"] == 0] == -100).all(),
      "padding never contributes to the loss")

In [ ]:
# The Trainer is Lab 1's loop plus mixed precision, gradient accumulation, scheduling, logging and
# checkpointing. Only the stable, long-lived arguments are used here so this cell survives
# transformers upgrades.
free_memory()
set_seed()

output_dir = "qwen-lora-emotion"
args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,   # effective batch = BATCH_SIZE * GRAD_ACCUM
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,              # ~10x a full fine-tune's LR: adapters start at zero
    lr_scheduler_type="cosine",
    warmup_steps=max(1, MAX_STEPS // 20),     # warmup_ratio is deprecated in transformers 5.2
    logging_steps=max(1, MAX_STEPS // 20),
    save_strategy="no",
    report_to=[],
    seed=SEED,
    fp16=USE_FP16,
    bf16=USE_BF16,
    remove_unused_columns=False,
    # The Trainer picks its own device unless told otherwise, and it will happily move the model
    # to MPS or CUDA behind your back. Everything after training then fails with
    # "Placeholder storage has not been allocated on MPS device" when we feed it CPU tensors.
    # Pin it to match DEVICE, and move the model back afterwards as a belt-and-braces measure.
    use_cpu=(DEVICE.type == "cpu"),
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized_train, data_collator=collator)
train_result = trainer.train()
model.to(DEVICE)

check(next(model.parameters()).device.type == DEVICE.type,
      f"the model is still on {DEVICE.type} after training - the Trainer did not relocate it")

log_steps = [(l["step"], l["loss"]) for l in trainer.state.log_history if "loss" in l]
print(f"\nfirst logged loss {log_steps[0][1]:.4f} -> final {log_steps[-1][1]:.4f}")
print(f"train runtime {train_result.metrics['train_runtime']:.1f}s "
      f"({train_result.metrics['train_samples_per_second']:.1f} samples/s)")

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot([s for s, _ in log_steps], [l for _, l in log_steps], "o-", ms=3)
ax.set_xlabel("step"); ax.set_ylabel("training loss"); ax.set_title("LoRA fine-tuning")
plt.tight_layout(); plt.show()

check(np.isfinite(log_steps[-1][1]), "the loss is finite - no fp16 overflow")
check(log_steps[-1][1] < log_steps[0][1], "the loss decreased")

## 6. Evaluation after fine-tuning

Same harness, same held-out examples, both methods. Nothing about the evaluation changed - which
is the only way the comparison means anything.

In [ ]:
score_preds_ft, score_matrix_ft = score_labels(model, eval_texts)
acc_score_ft = report(score_preds_ft, eval_labels, "fine-tuned, constrained scoring")

gen_preds_ft, gen_raw_ft = generate_labels(model, eval_texts)
acc_gen_ft = report(gen_preds_ft, eval_labels, "fine-tuned, free generation")

print("\n" + "=" * 62)
print(f"{'setting':<34} {'scoring':>10} {'generation':>12}")
print("-" * 62)
print(f"{'zero-shot (base)':<34} {acc_score_zs:>9.2%} {acc_gen_zs:>11.2%}")
print(f"{'6-shot (base)':<34} {acc_score_fs:>9.2%} {acc_gen_fs:>11.2%}")
print(f"{'LoRA fine-tuned':<34} {acc_score_ft:>9.2%} {acc_gen_ft:>11.2%}")
print("-" * 62)
print(f"{'majority class':<34} {majority:>9.2%}")
print("=" * 62)

check(acc_score_ft > acc_score_zs, "fine-tuning beats the zero-shot baseline")
check(acc_score_ft > acc_score_fs, "fine-tuning beats the 6-shot prompting baseline")
check(acc_score_ft > majority, "fine-tuning beats the majority-class baseline")

Note what happened to the **generation** column. Before fine-tuning it trailed the scoring column,
because the model wrote sentences instead of bare labels. After a couple of hundred steps the two
columns converge: the model learned the output format almost immediately, and the format tax is
gone. Format compliance is the cheapest thing a fine-tune buys, and often most of the headline gain.

In [ ]:
from collections import Counter


def confusion(preds, gold):
    m = np.zeros((len(LABELS), len(LABELS)), dtype=int)
    for g, p in zip(gold, preds):
        if p >= 0:
            m[g, p] += 1
    return m


cm_zs, cm_ft = confusion(score_preds_zs, eval_labels), confusion(score_preds_ft, eval_labels)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
for ax, cm, title, acc in [(axes[0], cm_zs, "zero-shot", acc_score_zs),
                           (axes[1], cm_ft, "LoRA fine-tuned", acc_score_ft)]:
    ax.imshow(cm, cmap="Blues")
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"{title} - {acc:.1%}")
plt.tight_layout(); plt.show()

print(f"{'label':<10} {'support':>8} {'recall zs':>11} {'recall ft':>11} {'delta':>8}")
for i, name in enumerate(LABELS):
    support = cm_ft[i].sum()
    if support == 0:
        continue
    r_zs, r_ft = cm_zs[i, i] / max(1, cm_zs[i].sum()), cm_ft[i, i] / support
    print(f"{name:<10} {support:>8} {r_zs:>10.1%} {r_ft:>10.1%} {r_ft - r_zs:>+7.1%}")

print(f"\nzero-shot prediction distribution: {dict(Counter(LABELS[p] for p in score_preds_zs))}")
print(f"fine-tuned prediction distribution: {dict(Counter(LABELS[p] for p in score_preds_ft))}")

The per-class table is where fine-tuning earns its keep. Zero-shot models on a fixed label set are
usually badly *miscalibrated* toward a couple of favoured labels - look at the prediction
distributions above. Fine-tuning largely fixes the prior even when it teaches little new semantics,
and that alone moves accuracy a long way.

## 7. Benchmarking

Accuracy is one of four numbers you owe an operations team. The others are latency, throughput and
footprint.

In [ ]:
adapter_dir = Path(output_dir) / "adapter"
model.save_pretrained(adapter_dir)

adapter_bytes = sum(f.stat().st_size for f in adapter_dir.rglob("*") if f.is_file())
base_bytes = total_params * 4
print(f"adapter on disk   {adapter_bytes / 1e6:8.2f} MB")
print(f"base model (fp32) {base_bytes / 1e9:8.2f} GB")
print(f"ratio             {base_bytes / adapter_bytes:8.0f}x smaller\n")
print("adapter contents:")
for f in sorted(adapter_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.name:<32} {f.stat().st_size / 1e6:8.3f} MB")

print(f"\n100 task-specific models: {100 * base_bytes / 1e9:.0f} GB as full fine-tunes, "
      f"{(base_bytes + 100 * adapter_bytes) / 1e9:.1f} GB as one base plus 100 adapters.")

check(adapter_bytes < base_bytes / 10,
      f"the adapter is an order of magnitude smaller than the base model "
      f"({base_bytes / adapter_bytes:.0f}x at r={LORA_R}; the ratio scales inversely with rank)")

In [ ]:
# Latency: base, adapter attached (unmerged), and adapter merged into the weights.
# Unmerged LoRA costs two extra small matmuls per adapted layer on every forward pass - real
# overhead in a latency-sensitive service. Merging removes it entirely, at the cost of pinning
# one adapter into the weights.
@torch.no_grad()
def bench_latency(m, batch_size=1, prompt_len=64, new_tokens=32, repeats=3):
    m.eval()
    ids = torch.randint(0, 1000, (batch_size, prompt_len), device=DEVICE)
    # Always pass an explicit attention mask to generate(). These synthetic prompts contain no
    # padding, but when the pad and EOS tokens coincide transformers cannot infer that and warns -
    # and on real padded input it would silently attend to the padding.
    mask = torch.ones_like(ids)
    gen_kwargs = dict(input_ids=ids, attention_mask=mask, do_sample=False,
                      pad_token_id=tokenizer.pad_token_id)

    m.generate(**gen_kwargs, max_new_tokens=4)                           # warm up
    times = []
    for _ in range(repeats):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        m.generate(**gen_kwargs, max_new_tokens=new_tokens)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    best = min(times)
    return best, batch_size * new_tokens / best


with model.disable_adapter():
    t_base, tps_base = bench_latency(model)
t_lora, tps_lora = bench_latency(model)

merged = model.merge_and_unload()      # returns a plain transformers model, adapters folded in
t_merged, tps_merged = bench_latency(merged)

print(f"{'configuration':<26} {'latency (s)':>12} {'tokens/s':>10} {'vs base':>9}")
print(f"{'base (adapter disabled)':<26} {t_base:>12.3f} {tps_base:>10.1f} {1.0:>8.2f}x")
print(f"{'LoRA attached (unmerged)':<26} {t_lora:>12.3f} {tps_lora:>10.1f} {t_base / t_lora:>8.2f}x")
print(f"{'LoRA merged':<26} {t_merged:>12.3f} {tps_merged:>10.1f} {t_base / t_merged:>8.2f}x")

# Merging must not change the outputs - verify before trusting the speed number.
merged_preds, _ = score_labels(merged, eval_texts[:64])
agreement = float((merged_preds == score_preds_ft[:64]).mean())
print(f"\nmerged vs unmerged agreement on 64 examples: {agreement:.1%}")
check(agreement >= 0.98, "merging the adapter preserves predictions")
check(t_merged <= t_lora * 1.15, "merged inference is no slower than unmerged")

In [ ]:
# Throughput against batch size. Latency is what one user feels; throughput is what you pay for.
batch_sizes = [1, 2, 4, 8]
rows = []
for bs in batch_sizes:
    lat, tps = bench_latency(merged, batch_size=bs, new_tokens=24, repeats=2)
    rows.append((bs, lat, tps))
    free_memory()

print(f"{'batch':>6} {'latency (s)':>12} {'tokens/s':>10} {'per-seq latency':>17}")
for bs, lat, tps in rows:
    print(f"{bs:>6} {lat:>12.3f} {tps:>10.1f} {lat:>16.3f}s")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot([r[0] for r in rows], [r[2] for r in rows], "o-")
axes[0].set_xlabel("batch size"); axes[0].set_ylabel("tokens/s"); axes[0].set_title("Throughput")
axes[1].plot([r[0] for r in rows], [r[1] for r in rows], "o-")
axes[1].set_xlabel("batch size"); axes[1].set_ylabel("seconds"); axes[1].set_title("Batch latency")
plt.tight_layout(); plt.show()

print(f"\nbatching {batch_sizes[-1]}x raised throughput {rows[-1][2] / rows[0][2]:.1f}x "
      f"while batch latency grew {rows[-1][1] / rows[0][1]:.1f}x.")
print("That gap is the entire economic argument for continuous batching in vLLM and TGI: decoding")
print("is memory-bandwidth bound, so extra sequences ride along nearly free until compute saturates.")

if DEVICE.type == "cuda":
    print(f"\npeak CUDA memory {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

check(rows[-1][2] > rows[0][2], "larger batches deliver more total throughput")

## 8. Shipping the adapter

The adapter is the deliverable. It loads onto any copy of the base model, which is what makes
adapter-per-tenant serving practical - `PeftModel.from_pretrained` on a shared base, or
`add_adapter`/`set_adapter` to hold several in memory and switch per request.

In [ ]:
del merged
free_memory()

reloaded_base = load_causal_lm(MODEL_ID).to(DEVICE)
reloaded = PeftModel.from_pretrained(reloaded_base, adapter_dir).eval()

reload_preds, _ = score_labels(reloaded, eval_texts[:64])
reload_agreement = float((reload_preds == score_preds_ft[:64]).mean())
print(f"reloaded-from-disk agreement with the in-memory model: {reload_agreement:.1%}")

print("\nadapter_config.json:")
print(json.dumps(json.loads((adapter_dir / "adapter_config.json").read_text()), indent=2)[:700])

check(reload_agreement >= 0.98, "a reloaded adapter reproduces the fine-tuned model")

# Upload with `model.push_to_hub("<user>/qwen2.5-0.5b-emotion-lora")` after `huggingface_hub.login()`.
# Not executed here - it needs credentials and publishes to a public namespace.

In [ ]:
del reloaded, reloaded_base
free_memory()
shutil.rmtree(output_dir, ignore_errors=True)

## 9. Exercises

1. **Rank sweep.** Train at `r` in `{1, 4, 16, 64}` with `alpha = 2r`. Plot accuracy against
   trainable parameters. The curve usually saturates early - find where, and decide what rank you
   would actually ship.
2. **Target modules.** Compare attention-only (`q_proj`, `v_proj`) against all seven projections at
   matched *trainable parameter count* (raise `r` for the attention-only run). Which placement wins
   per parameter?
3. **Learning rate.** LoRA typically wants ~10x a full fine-tune's rate. Sweep `{2e-5, 2e-4, 2e-3}`
   and explain the failure at each end.
4. **Delete the loss mask.** Set `labels = input_ids` and retrain. Watch the loss curve look
   *better* (it is averaging over easy prompt tokens) while accuracy drops. A textbook example of
   a training metric that is not the metric you care about.
5. **Catastrophic forgetting.** Ask the fine-tuned model a general question ("What is the capital
   of France?"). Does it still answer, or does it emit an emotion label? Then compare `r=64` at
   1000 steps against `r=4` at 200 steps - which forgets more, and why?
6. **Data scaling.** Train on `N_TRAIN` in `{100, 500, 2000, 8000}` at a fixed step count. Plot
   accuracy against data volume, and find the point where more data stops paying.
7. **QLoRA.** Load the base in 4-bit with `bitsandbytes`
   (`BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)`) and repeat.
   Compare memory, speed, and accuracy. Requires a CUDA GPU.

## What you built, and what comes next

* Mapped a real pretrained model onto the architecture you implemented in Lab 2.
* Built an evaluation that separates *knowing the answer* from *formatting the answer*, and a
  few-shot baseline strong enough to make the fine-tune prove itself.
* Implemented LoRA and verified it against PEFT numerically.
* Trained with completion-only loss masking, then measured accuracy, per-class recall, latency
  merged and unmerged, throughput against batch size, and artifact size on disk.

**Lab 4** takes the same model and the same task into **JAX and Ray**: a Qwen2 forward pass written
in JAX and verified logit-for-logit against these HuggingFace weights, LoRA training under
`jit`/`grad`/`vmap`, and a Ray-parallel hyperparameter sweep that runs the rank ablation from
exercise 1 across workers instead of one at a time.